# Process, in code

Runs the analysis one numbered section at a time, so each result can be
seen as it is computed.

Nothing is re-implemented. Every cell calls the same functions the pipeline
uses (`src/candy/`), so what you see is what `python -m candy` computes. The
last cell proves it by rebuilding the whole results bundle and comparing it
with the committed one.

**To run:** from the repository root, after `uv sync`:

```sh
uv run --with jupyter jupyter lab PROCESS_CODE.ipynb
```

Run the cells top to bottom; later cells use variables from earlier ones.

## 0. The big picture

Setup: imports and the two input files.

In [1]:
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from candy.data import ATTRIBUTES, PREDICTORS, Interaction, load_candies
from candy.estimators import EstimationError, fit, holm
from candy.pipeline import ALPHA, BOOTSTRAP_DRAWS, INTERACTIONS, SEED, run
from candy.profiles import (
    ProfileScorer, all_profiles, benchmarks, choose_levels, compare_cookie_and_gummy,
    evidence_tiers, is_realistic, is_supported, mark_ties, nearby_candies, rank, recommend,
)

pd.set_option("display.precision", 3, "display.width", 120)

ROOT = Path.cwd()  # the repository root
RAW = ROOT / "data" / "raw" / "candy-data.csv"
GUMMY = ROOT / "data" / "derived" / "gummy.csv"


def names_on(attributes):
    """The attributes a profile has switched on, as readable text."""
    return " + ".join(a for a, on in attributes.items() if on) or "(none)"

## 1. The data and how it was labelled

### 1.1 Where the data comes from

The raw file exactly as delivered: 85 rows, one per survey entry.

In [2]:
raw = pd.read_csv(RAW)
print(raw.shape)
raw.head()

(85, 13)


,competitorname,chocolate,fruity,caramel,peanutyalmondy,nougat,crispedricewafer,hard,bar,pluribus,sugarpercent,pricepercent,winpercent
0,100 Grand,1,0,1,0,0,1,0,1,0,0.732,0.860,66.972
1,3 Musketeers,1,0,0,0,1,0,0,1,0,0.604,0.511,67.603
2,One dime,0,0,0,0,0,0,0,0,0,0.011,0.116,32.261
3,One quarter,0,0,0,0,0,0,0,0,0,0.011,0.511,46.117
4,Air Heads,0,1,0,0,0,0,0,0,0,0.906,0.511,52.341


`winpercent` is the win rate. The best and worst entries:

In [3]:
raw.sort_values("winpercent", ascending=False)[["competitorname", "winpercent"]].iloc[[0, 1, 2, -3, -2, -1]]

,competitorname,winpercent
52,Reese's Peanut Butter cup,84.180
51,Reese's Miniatures,81.866
79,Twix,81.643
12,Chiclets,24.525
7,Boston Baked Beans,23.418
44,Nik L Nip,22.445


### 1.2 The labels that came with the data

One candy can have several attributes. Snickers has five; its sugar level of 0.546 means it is more sugary than about 55% of the entries.

In [4]:
raw[raw["competitorname"] == "Snickers"].T

,64
competitorname,Snickers
chocolate,1
fruity,0
caramel,1
peanutyalmondy,1
nougat,1
crispedricewafer,0
hard,0
bar,1
pluribus,0


How common is each attribute? Rare ones (nougat, crisped rice/wafer) will be hard to measure.

In [5]:
raw.drop(columns=["competitorname", "sugarpercent", "pricepercent", "winpercent"]).sum().sort_values()

nougat               7
crispedricewafer     7
caramel             14
peanutyalmondy      14
hard                15
bar                 21
chocolate           37
fruity              38
pluribus            44
dtype: int64

### 1.3 The label we added ourselves: gummy

The hand-coded table: one row per candy, with the reason and a flag on judgement calls.

In [6]:
gummy_table = pd.read_csv(GUMMY)
gummy_table.head()

,candy,gummy,unclear,note
0,100 Grand,0,0,chocolate bar with caramel and crisped rice
1,3 Musketeers,0,0,chocolate-covered nougat bar
2,Air Heads,0,0,taffy texture; not a gummy chew
3,Almond Joy,0,0,chocolate coconut almond bar
4,Baby Ruth,0,0,chocolate peanut nougat caramel bar


How many candies fall in each cell (rows: gummy or not; columns: clear or judgement call):

In [7]:
pd.crosstab(gummy_table["gummy"], gummy_table["unclear"])

unclear,0,1
gummy,,
0,65,7
1,7,4


Every gummy and every judgement call, with the note explaining the decision:

In [8]:
with pd.option_context("display.max_colwidth", None):
    display(gummy_table[(gummy_table["gummy"] == 1) | (gummy_table["unclear"] == 1)])

,candy,gummy,unclear,note
6,Candy Corn,0,1,mellowcreme; some recipes use gelatin but texture is waxy fondant
9,Chewey Lemonhead Fruit Mix,0,1,shelled jelly candy like Mike & Ike; firmer than a gummy
11,Dots,1,1,starch-based gumdrop; gummy texture without gelatin
13,Fruit Chews,0,1,generic name; most fruit chews are taffy-style
16,Haribo Gold Bears,1,0,classic gelatin gummy bear
17,Haribo Happy Cola,1,0,gelatin gummy
18,Haribo Sour Bears,1,0,sour gelatin gummy bear
19,Haribo Twin Snakes,1,0,gelatin gummy
29,Lifesavers big ring gummies,1,0,gelatin gummy rings
32,Mike & Ike,0,1,shelled jelly candy; firmer than a gummy


### 1.4 Cleaning

`load_candies` renames, validates, drops the coins and attaches the gummy label.

In [9]:
data = load_candies(RAW, GUMMY)
candies = data.candies
print("raw entries:", data.n_raw, "| excluded:", data.excluded, "| candies analysed:", len(candies))
candies.head(3)

raw entries: 85 | excluded: ['One dime', 'One quarter'] | candies analysed: 83


,candy,chocolate,fruity,caramel,peanuty_almondy,nougat,crisped_rice_wafer,hard,bar,pluribus,sugar_level,price_level,win_rate,gummy,gummy_unclear
0,100 Grand,1,0,1,0,0,1,0,1,0,0.732,0.860,66.972,0,0
1,3 Musketeers,1,0,0,0,1,0,0,1,0,0.604,0.511,67.603,0,0
2,Air Heads,0,1,0,0,0,0,0,0,0,0.906,0.511,52.341,0,0


The two coins: no attributes at all, and a win rate that measures something other than taste.

In [10]:
raw[raw["competitorname"].isin(data.excluded)]

,competitorname,chocolate,fruity,caramel,peanutyalmondy,nougat,crispedricewafer,hard,bar,pluribus,sugarpercent,pricepercent,winpercent
2,One dime,0,0,0,0,0,0,0,0,0,0.011,0.116,32.261
3,One quarter,0,0,0,0,0,0,0,0,0,0.011,0.511,46.117


Validation stops the run instead of repairing. Here a candy is listed twice on purpose:

In [11]:
lines = RAW.read_text().splitlines()
with tempfile.TemporaryDirectory() as tmp:
    broken = Path(tmp) / "repeated.csv"
    broken.write_text("\n".join([*lines, lines[1]]) + "\n")
    try:
        load_candies(broken, GUMMY)
    except ValueError as error:
        print("ValueError:", error)

ValueError: raw data lists candies more than once: ['100 Grand']


## 2. The model

### 2.1 Why a model is needed at all

The naive approach: average win rate with and without each attribute. Look at fruity: it seems to *hurt* by about 12 points.

In [12]:
naive = pd.DataFrame({
    a: {"with": candies.loc[candies[a] == 1, "win_rate"].mean(),
        "without": candies.loc[candies[a] == 0, "win_rate"].mean()}
    for a in ATTRIBUTES
}).T
naive["difference"] = naive["with"] - naive["without"]
naive.sort_values("difference")

,with,without,difference
hard,40.509,52.808,-12.299
fruity,44.120,56.044,-11.925
pluribus,46.823,54.829,-8.007
caramel,57.347,49.213,8.134
nougat,60.052,49.713,10.339
bar,61.295,46.957,14.338
peanuty_almondy,63.697,47.924,15.773
crisped_rice_wafer,66.170,49.149,17.021
chocolate,60.922,42.271,18.651


The reason is confounding: features travel together. Among non-fruity candies, most are chocolate.

In [13]:
pd.crosstab(candies["fruity"], candies["chocolate"])

chocolate,0,1
fruity,,
0,9,36
1,37,1


![](outputs/figures/attribute_comparison.png)

### 2.2 What the model looks like

Fit the main model: win rate on the nine attributes plus sugar and price level. The `coef` column is the points each feature adds.

In [14]:
X = candies[PREDICTORS].to_numpy()
y = candies["win_rate"].to_numpy()
main_model = fit(X, y, PREDICTORS, cov="hc3", alpha=ALPHA)
main_model.table()

,coef,se,t,p,ci_low,ci_high
intercept,32.017,5.365,5.968,8.598e-08,21.319,42.715
chocolate,21.214,4.633,4.578,1.946e-05,11.975,30.453
fruity,11.008,4.395,2.505,1.455e-02,2.245,19.771
caramel,2.793,4.740,0.589,5.576e-01,-6.658,12.243
peanuty_almondy,10.696,5.310,2.014,4.775e-02,0.109,21.284
nougat,0.413,7.262,0.057,9.548e-01,-14.066,14.893
crisped_rice_wafer,8.885,7.519,1.182,2.413e-01,-6.107,23.876
hard,-5.864,3.134,-1.871,6.546e-02,-12.112,0.385
bar,1.693,6.317,0.268,7.895e-01,-10.902,14.288
pluribus,0.201,3.361,0.060,9.525e-01,-6.501,6.904


"Least squares" is nothing mysterious. The same coefficients in two lines of NumPy:

In [15]:
design = np.column_stack([np.ones(len(X)), X])  # a column of ones gives the intercept
by_hand, *_ = np.linalg.lstsq(design, y, rcond=None)
print(np.allclose(by_hand, main_model.coef))

True


The worked example from the document: a plain chocolate product at median sugar and price.

In [16]:
levels = choose_levels(candies)
b = main_model.coef
print(levels)
print(f"{b['intercept']:.1f} + {b['chocolate']:.1f} "
      f"+ {b['sugar_level']:.1f}*{levels['sugar_level']} "
      f"+ ({b['price_level']:.1f})*{levels['price_level']} "
      f"= {b['intercept'] + b['chocolate'] + b['sugar_level'] * levels['sugar_level'] + b['price_level'] * levels['price_level']:.1f}")

{'sugar_level': 0.465, 'price_level': 0.465}
32.0 + 21.2 + 10.2*0.465 + (-7.3)*0.465 = 54.6


### 2.3 A trap when reading chocolate and fruity

Only a handful of candies are neither chocolate nor fruity, and both coefficients are measured against them:

In [17]:
neither = candies[(candies["chocolate"] == 0) & (candies["fruity"] == 0)]
print(len(neither), "candies:", neither["candy"].tolist())

9 candies: ['Boston Baked Beans', 'Candy Corn', 'Haribo Happy Cola', 'Payday', 'Pixie Sticks', 'Root Beer Barrels', 'Sugar Babies', 'Sugar Daddy', "Werther's Original Caramel"]


So the useful number is the direct comparison, chocolate minus fruity, everything else equal:

In [18]:
scorer = ProfileScorer(main_model, levels, ())
blank = dict.fromkeys(ATTRIBUTES, 0)
scorer.gap(blank | {"chocolate": 1}, blank | {"fruity": 1})

Difference(estimate=10.206069618642921, ci_low=np.float64(1.3884043076758346), ci_high=np.float64(19.023734929610008))

### 2.4 How sure are we? Standard errors, intervals and p-values

`se` is the standard error, `ci_low`/`ci_high` the 95% confidence interval, `p` the p-value (all in the table above). To see what HC3 changes, compare with the textbook ("classical") standard errors:

In [19]:
classical = fit(X, y, PREDICTORS, cov="classical", alpha=ALPHA)
pd.DataFrame({"classical se": classical.se, "HC3 se": main_model.se,
              "ratio": main_model.se / classical.se})

,classical se,HC3 se,ratio
intercept,5.050,5.365,1.062
chocolate,4.119,4.633,1.125
fruity,4.068,4.395,1.080
caramel,3.688,4.740,1.285
peanuty_almondy,3.648,5.310,1.455
nougat,5.713,7.262,1.271
crisped_rice_wafer,5.260,7.519,1.429
hard,3.466,3.134,0.904
bar,5.151,6.317,1.226
pluribus,3.190,3.361,1.054


![](outputs/figures/coefficients.png)

### 2.5 How good is the model? R² and leave-one-out

In-sample R² against the fair, leave-one-out version:

In [20]:
print(f"R²               {main_model.r_squared:.3f}")
print(f"leave-one-out R² {main_model.loo_r_squared:.3f}")

R²               0.546
leave-one-out R² 0.351


The package uses a shortcut formula for leave-one-out. Here it is done the slow, literal way: drop each candy, refit on the other 82, predict the dropped one.

In [21]:
errors = []
for i in range(len(y)):
    keep = np.arange(len(y)) != i
    refit = fit(X[keep], y[keep], PREDICTORS)
    prediction = refit.coef["intercept"] + X[i] @ refit.coef[PREDICTORS].to_numpy()
    errors.append(y[i] - prediction)
slow = 1 - np.sum(np.square(errors)) / np.sum((y - y.mean()) ** 2)
print(f"slow leave-one-out R² {slow:.3f}  (matches: {np.isclose(slow, main_model.loo_r_squared)})")

slow leave-one-out R² 0.351  (matches: True)


### 2.6 Do the features overlap too much? VIF

1 means no overlap; above 5 to 10 is the usual warning zone.

In [22]:
main_model.vif.sort_values(ascending=False)

bar                   3.646
chocolate             3.047
fruity                2.987
pluribus              1.843
price_level           1.837
nougat                1.832
crisped_rice_wafer    1.553
caramel               1.386
peanuty_almondy       1.357
hard                  1.293
sugar_level           1.227
dtype: float64

### 2.7 Testing many things at once: the Holm correction

`p_holm` is the adjusted p-value. Only effects with `p_holm` below 0.05 survive.

In [23]:
p = main_model.p[PREDICTORS]
holm_table = pd.DataFrame({"coef": main_model.coef[PREDICTORS], "p": p, "p_holm": holm(p)})
holm_table["survives"] = holm_table["p_holm"] < ALPHA
holm_table.sort_values("p")

,coef,p,p_holm,survives
chocolate,21.214,1.946e-05,2.141e-04,True
fruity,11.008,1.455e-02,1.455e-01,False
sugar_level,10.161,4.222e-02,3.800e-01,False
peanuty_almondy,10.696,4.775e-02,3.820e-01,False
hard,-5.864,6.546e-02,4.582e-01,False
crisped_rice_wafer,8.885,2.413e-01,1.000e+00,False
price_level,-7.281,2.608e-01,1.000e+00,False
caramel,2.793,5.576e-01,1.000e+00,False
bar,1.693,7.895e-01,1.000e+00,False
pluribus,0.201,9.525e-01,1.000e+00,False


The evidence tier of each attribute follows from that:

In [24]:
survivors = set(holm_table.index[holm_table["survives"]])
evidence = evidence_tiers(main_model, confirmed=survivors)
pd.DataFrame(evidence).T

,tier,direction
chocolate,driver,positive
fruity,tentative,positive
caramel,open,positive
peanuty_almondy,tentative,positive
nougat,open,positive
crisped_rice_wafer,open,positive
hard,open,negative
bar,open,positive
pluribus,open,positive


### 2.8 Why the estimators are built from NumPy and SciPy, and how we trust them

The two safety checks. First, a feature that is an exact copy of another cannot be separated from it:

In [25]:
copy = np.column_stack([X, candies["chocolate"]])
try:
    fit(copy, y, [*PREDICTORS, "chocolate_again"])
except EstimationError as error:
    print("EstimationError:", error)

EstimationError: design matrix is rank deficient: some terms are exact linear combinations of others and cannot be estimated separately


The second check (one candy alone fixing an estimate) shows up in the next section.

## 3. Do features work better together? Interactions

Each declared interaction is added to the main model on its own. The counts show how many candies there are to learn from.

In [26]:
rows = []
for interaction in INTERACTIONS:
    has_a = candies[interaction.attribute] == 1
    has_b = candies[interaction.partner] == 1
    row = {"interaction": interaction.term, "both": int((has_a & has_b).sum()),
           "partner only": int((~has_a & has_b).sum()), "chocolate only": int((has_a & ~has_b).sum())}
    with_term = candies[PREDICTORS].assign(**{interaction.term: interaction.values(candies)})
    try:
        model = fit(with_term.to_numpy(), y, list(with_term.columns), cov="hc3", alpha=ALPHA)
        row |= {"coef": model.coef[interaction.term], "ci_low": model.ci_low[interaction.term],
                "ci_high": model.ci_high[interaction.term],
                "loo R² change": model.loo_r_squared - main_model.loo_r_squared}
    except EstimationError as error:
        row["problem"] = str(error)[:45] + "..."
    rows.append(row)
pd.DataFrame(rows).set_index("interaction")

,both,partner only,chocolate only,coef,ci_low,ci_high,loo R² change,problem
interaction,,,,,,,,
chocolate_x_peanuty_almondy,12,2,25,14.417,-19.5,48.334,-0.027,NaN
chocolate_x_crisped_rice_wafer,7,0,30,NaN,NaN,NaN,NaN,design matrix is rank deficient: some terms a...
chocolate_x_bar,20,1,17,NaN,NaN,NaN,NaN,an observation has leverage 1 (it alone ident...


None is kept (wide interval and worse predictions, or not estimable), so the final model is the main model.

## 4. Would a different choice change the answer? Robustness checks

**Bootstrap.** Refit on 2,000 random re-draws of the candies and compare the resulting intervals with HC3.

In [27]:
booted = fit(X, y, PREDICTORS, cov="hc3", alpha=ALPHA, bootstrap_draws=BOOTSTRAP_DRAWS, seed=SEED)
pd.DataFrame({"HC3 low": main_model.ci_low, "HC3 high": main_model.ci_high,
              "bootstrap low": booted.bootstrap.ci_low, "bootstrap high": booted.bootstrap.ci_high}).loc[PREDICTORS]

,HC3 low,HC3 high,bootstrap low,bootstrap high
chocolate,11.975,30.453,13.123,30.392
fruity,2.245,19.771,3.985,19.706
caramel,-6.658,12.243,-5.493,11.794
peanuty_almondy,0.109,21.284,2.360,20.401
nougat,-14.066,14.893,-12.831,13.408
crisped_rice_wafer,-6.107,23.876,-1.712,23.674
hard,-12.112,0.385,-11.834,-0.465
bar,-10.902,14.288,-9.938,13.398
pluribus,-6.501,6.904,-5.688,7.047
sugar_level,0.367,19.955,0.883,19.074


**Logit model.** Same predictors, win rate transformed so predictions cannot leave 0 to 100. Effects converted back to points, next to the main model's:

In [28]:
logit_model = fit(X, y, PREDICTORS, cov="hc3", alpha=ALPHA, target="logit")
pd.DataFrame({"linear model": main_model.coef[PREDICTORS],
              "logit model, in points": logit_model.effects_in_points()["coef"]})

,linear model,"logit model, in points"
chocolate,21.214,22.049
fruity,11.008,10.445
caramel,2.793,2.770
peanuty_almondy,10.696,11.297
nougat,0.413,0.625
crisped_rice_wafer,8.885,9.547
hard,-5.864,-5.864
bar,1.693,0.703
pluribus,0.201,-0.490
sugar_level,10.161,10.429


**Coins included** and **without Reese's**: how far does each coefficient move?

In [29]:
everything = data.with_coins
coins_model = fit(everything[PREDICTORS].to_numpy(), everything["win_rate"].to_numpy(), PREDICTORS, cov="hc3")
no_reeses = candies[~candies["candy"].str.startswith("Reese's")]
brand_model = fit(no_reeses[PREDICTORS].to_numpy(), no_reeses["win_rate"].to_numpy(), PREDICTORS, cov="hc3")
print("largest shift with coins:", round((coins_model.coef - main_model.coef)[PREDICTORS].abs().max(), 2))
pd.DataFrame({"main": main_model.coef, "with coins": coins_model.coef,
              "without Reese's": brand_model.coef, "p without Reese's": brand_model.p}).loc[PREDICTORS]

largest shift with coins:

 1.59


,main,with coins,without Reese's,p without Reese's
chocolate,21.214,19.748,17.953,4.398e-05
fruity,11.008,9.422,11.480,5.353e-03
caramel,2.793,2.224,4.550,3.076e-01
peanuty_almondy,10.696,10.071,4.262,4.455e-01
nougat,0.413,0.804,0.079,9.909e-01
crisped_rice_wafer,8.885,8.919,7.983,2.060e-01
hard,-5.864,-6.165,-4.699,1.328e-01
bar,1.693,0.442,10.129,8.561e-02
pluribus,0.201,-0.854,4.222,1.704e-01
sugar_level,10.161,9.087,9.684,6.747e-02


Chocolate holds everywhere. Peanut/almond roughly halves without Reese's.

![](outputs/figures/robustness.png)

## 5. From model to recommendation

### 5.1 Listing candidate products

512 combinations, filtered twice.

In [30]:
profiles = all_profiles()
realistic = is_realistic(profiles)
supported = realistic & is_supported(profiles, candies)
print("all:", len(profiles), "| realistic:", int(realistic.sum()), "| supported:", int(supported.sum()))

all: 512 | realistic: 240 | supported: 109


### 5.2 Scoring each profile

Every supported profile is predicted at median sugar and price. `ci_*` is the range for the average, `pi_*` the wider range for one launch.

In [31]:
ranked = rank(profiles[supported], scorer)
ranking = pd.DataFrame([{"rank": r["rank"], "profile": names_on(r["attributes"]),
                         **{k: r[k] for k in ("estimate", "ci_low", "ci_high", "pi_low", "pi_high")}}
                        for r in ranked]).set_index("rank")
with pd.option_context("display.max_colwidth", None):
    display(ranking.head(5))

,profile,estimate,ci_low,ci_high,pi_low,pi_high
rank,,,,,,
1,chocolate + caramel + peanuty_almondy + nougat + crisped_rice_wafer + bar,79.050,58.025,100.000,49.117,100.0
2,chocolate + caramel + peanuty_almondy + crisped_rice_wafer + bar,78.636,57.004,100.000,48.273,100.0
3,chocolate + caramel + peanuty_almondy + crisped_rice_wafer,76.944,54.555,99.333,46.037,100.0
4,chocolate + peanuty_almondy + crisped_rice_wafer + bar,75.844,58.115,93.572,48.126,100.0
5,chocolate + peanuty_almondy + crisped_rice_wafer + pluribus,74.352,54.473,94.231,45.212,100.0


### 5.3 Why the top-scoring profile is not recommended

The top profile stacks six attributes, five of them without real evidence. The recommendation is the best profile made only of drivers:

In [32]:
recommendation = recommend(ranked, candies, evidence)
primary = recommendation["primary"]
print(recommendation["rule"])
print()
print("primary profile:", names_on(primary["attributes"]), "| rank", primary["rank"], "of", len(ranked))
print(f"predicted {primary['estimate']:.1f}, average between {primary['ci_low']:.1f} and {primary['ci_high']:.1f}, "
      f"one launch between {primary['pi_low']:.1f} and {primary['pi_high']:.1f}")

The primary profile is the best supported profile made only of drivers: attributes whose positive effect survives the multiple-testing correction. Every other attribute is a design choice the data does not decide.

primary profile: chocolate | rank 48 of 109
predicted 54.6, average between 44.4 and 64.7, one launch between 31.0 and 78.2


### 5.4 Which profiles are really different from the recommendation?

Each profile's gap to the primary one is tested: tied, above or below.

In [33]:
ranked = mark_ties(ranked, scorer, primary["attributes"])
pd.Series([r["vs_primary"] or "(the primary itself)" for r in ranked]).value_counts()

tied                    88
below                   19
above                    1
(the primary itself)     1
Name: count, dtype: int64

The puzzle: the one profile "above" is ranked lower than 21 profiles that are only "tied". Its gap is one coefficient; theirs are sums of several uncertain ones, so their intervals are wider.

In [34]:
gaps = []
for r in ranked[:23]:
    if r["attributes"] == primary["attributes"]:
        continue
    gap = scorer.gap(r["attributes"], primary["attributes"])
    gaps.append({"rank": r["rank"], "profile": names_on(r["attributes"]), "gap": gap.estimate,
                 "ci_low": gap.ci_low, "ci_high": gap.ci_high, "vs primary": r["vs_primary"]})
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(gaps).set_index("rank").iloc[[0, 1, 2, 19, 20, 21, 22]])

,profile,gap,ci_low,ci_high,vs primary
rank,,,,,
1,chocolate + caramel + peanuty_almondy + nougat + crisped_rice_wafer + bar,24.480,-0.364,49.324,tied
2,chocolate + caramel + peanuty_almondy + crisped_rice_wafer + bar,24.066,-3.365,51.497,tied
3,chocolate + caramel + peanuty_almondy + crisped_rice_wafer,22.374,-2.014,46.762,tied
20,chocolate + nougat + crisped_rice_wafer + bar,10.991,-8.673,30.654,tied
21,chocolate + peanuty_almondy + pluribus,10.897,-2.218,24.013,tied
22,chocolate + peanuty_almondy,10.696,0.109,21.284,above
23,chocolate + crisped_rice_wafer + bar,10.577,-5.954,27.108,tied


![](outputs/figures/ranked_profiles.png)

### 5.5 A reality check against real candies

Real candies within one attribute of the primary profile, and the three named benchmarks:

In [35]:
print(nearby_candies(primary["attributes"], candies))
pd.DataFrame(benchmarks(primary["attributes"], candies))

{'max_attributes_differing': 1, 'n': 14, 'mean_win_rate': 56.62346071428572, 'min_win_rate': 34.722, 'max_win_rate': 84.18029}


,candy,attributes_differing,win_rate
0,Sixlets,1,34.722
1,Hershey's Kisses,1,55.375
2,Reese's Peanut Butter cup,1,84.180


## 6. Cookie versus gummy

The gummy model is the main model plus the hand-coded gummy label.

In [36]:
with_gummy = [*PREDICTORS, "gummy"]
gummy_model = fit(candies[with_gummy].to_numpy(), y, with_gummy, cov="hc3", alpha=ALPHA)
gummy_model.table().loc[["chocolate", "fruity", "crisped_rice_wafer", "gummy"]]

,coef,se,t,p,ci_low,ci_high
chocolate,22.733,4.880,4.659,1.473e-05,13.001,32.465
fruity,9.079,4.569,1.987,5.080e-02,-0.033,18.191
crisped_rice_wafer,8.968,7.521,1.192,2.372e-01,-6.033,23.968
gummy,9.450,3.964,2.384,1.984e-02,1.544,17.356


The three products of the debate, then the two direct differences:

In [37]:
debate = compare_cookie_and_gummy(ProfileScorer(gummy_model, levels, (), extra_attributes=("gummy",)))
display(pd.DataFrame(debate["products"]).T[["estimate", "ci_low", "ci_high"]])
pd.DataFrame(debate["differences"]).T

,estimate,ci_low,ci_high
chocolate,55.067,44.957,65.176
chocolate_cookie,64.034,46.717,81.352
gummy,50.863,43.367,58.359


,estimate,ci_low,ci_high,decided
chocolate_cookie_minus_gummy,13.171,-3.935,30.278,False
chocolate_minus_gummy,4.204,-4.668,13.076,False


`decided` is False for both: the leads favour chocolate, but the intervals include zero.

How much does the gummy effect depend on the judgement calls? Recode them as not gummy and refit:

In [38]:
clear_only = candies.assign(gummy=candies["gummy"] * (1 - candies["gummy_unclear"]))
clear_model = fit(clear_only[with_gummy].to_numpy(), y, with_gummy, cov="hc3", alpha=ALPHA)
pd.DataFrame({"all 11 gummies": gummy_model.table().loc["gummy"],
              "7 clear gummies": clear_model.table().loc["gummy"]}).T[["coef", "ci_low", "ci_high", "p"]]

,coef,ci_low,ci_high,p
all 11 gummies,9.450,1.544,17.356,0.02
7 clear gummies,5.558,-1.683,12.799,0.13


![](outputs/figures/cookie_vs_gummy.png)

## 7. What the results mean, and what they do not

One limit is worth seeing in numbers: how well does each model predict candies it has not seen?

In [39]:
chocolate_only = fit(candies[["chocolate"]].to_numpy(), y, ["chocolate"], cov="hc3")
pd.Series({"chocolate only (1 predictor)": chocolate_only.loo_r_squared,
           "main model (11 predictors)": main_model.loo_r_squared,
           "gummy model (12 predictors)": gummy_model.loo_r_squared}, name="leave-one-out R²")

chocolate only (1 predictor)    0.369
main model (11 predictors)      0.351
gummy model (12 predictors)     0.381
Name: leave-one-out R², dtype: float64

Knowing only "is it chocolate?" predicts as well as the full main model. The other labels help explain, not forecast.

And the typical miss on a single candy, which is why the one-launch ranges are wide:

In [40]:
print(f"residual standard deviation: {np.sqrt(main_model.residual_variance):.1f} points")

residual standard deviation: 10.7 points


![](outputs/figures/win_rate_distribution.png)

## 8. Where each thing lives

Everything above is collected by `pipeline.run()` into the results bundle. Rebuild it into a temporary folder and compare with the committed file: same seed, same bundle.

In [41]:
committed = json.loads((ROOT / "outputs" / "results.json").read_text())
with tempfile.TemporaryDirectory() as tmp:
    rebuilt = run(raw_csv=RAW, gummy_csv=GUMMY, out_dir=Path(tmp))
    rebuilt = json.loads((Path(tmp) / "results.json").read_text())
print("sections:", list(rebuilt))
print("identical to outputs/results.json:", rebuilt == committed)

sections: ['seed', 'sample', 'main_model', 'holm', 'chocolate_vs_fruity', 'attribute_comparison', 'interactions', 'final_model', 'profiles', 'recommendation', 'cookie_vs_gummy', 'robustness', 'data_notes']
identical to outputs/results.json: True
